# Weight-space programming on Snowflake

A frontier **writer** LM, called through Snowflake `AI_COMPLETE`, repairs a small trained classifier by editing its weights directly, with no gradients and no training. The protocol follows [dspy-wsp](https://github.com/cmpnd-ai/dspy-wsp) and *Sparks of Weight-Space Programming* (Takehi, Teoh, Zhang, Khattab, 2026):

| dspy-wsp | This notebook |
|---|---|
| Multi-turn writer with an explanation and one action per turn | Same; a turn without an explanation is returned unrun, at most twice in a row |
| Budgeted forward-pass tool; refused requests are free | `generate` and `activations` cost 1 pass each; `inspect` is free but capped |
| Paired original-vs-edited scoring with standard errors | `weights="both"` returns the paired gain ± SE |
| Practice / holdout / hidden validation | Holdout returns aggregates only; validation is scored once at the end |
| Guardrails: no edits to embeddings, output layer or norm, no whole-tensor rescales | Same, and they are re-checked before every pass |
| Patch with hashes and `write_checkpoint` | Patch with per-tensor SHA-256 values and a hash-checked `apply_patch` |
| Writer code runs in a sandboxed kernel | **No code execution.** The writer sends JSON edit operations (`project_out`, `add_rank1`, `set_entries`, `drop_edit`) that the host validates |

**The student** is a NumPy MLP (10 → 32 → 32 → 3) trained in-notebook to route tickets to billing, technical or account. Three of its ten features (`caps_ratio`, `emoji_count`, `ticket_length`) never affect the labels. We then **plant a failure**: a rank-1 shortcut in `block1` that turns the `caps_ratio` signal into a push toward billing. The writer is not told what the failure is.

`weight_space.py` must be in the same Workspace folder. This is a teaching harness with a known ground truth, not evidence about editing production LMs.

In [ ]:
import json
import platform
import numpy as np
from snowflake import snowpark
import importlib
import weight_space as ws
ws = importlib.reload(ws)  # pick up the current weight_space.py if an older copy is already loaded
assert hasattr(ws, 'build'), f'stale weight_space loaded from {ws.__file__}; restart the kernel'

session = snowpark.context.get_active_session()
harness, clean, truth = ws.build(seed=7, max_passes=12, max_inspects=40, max_turns=40)

def accuracy(weights, split):
    x, y = split
    return float((ws.forward(weights, x).argmax(1) == y).mean())

print('Base fingerprint:', harness.base_hash)
print(f"Clean student on validation:  {accuracy(clean, harness._valset):.3f}")
print(f"Broken student on validation: {accuracy(harness.base, harness._valset):.3f}")
print('Weights:', {n: list(harness.base[n].shape) for n in ws.TENSORS})

## Controls (free, no inference)

These run before any model call:

1. **Scripted control:** a fixed policy that finds the outlier singular direction of `block1`, projects it out, and checks the result on the holdout. It is not model output; it shows that the tools are sufficient.
2. **Guardrail checks:** an edit to a protected tensor and a whole-tensor rescale must both be refused.
3. **Negative control:** projecting out a random direction should not repair the model.

In [ ]:
control, _, _ = ws.build(seed=7)
ws.run_writer(control, ws.scripted_writer(control))
print('Scripted control validation:', control.validate()['validation'])

probe, _, _ = ws.build(seed=7)
for bad in ({'tool': 'edit', 'op': 'set_entries', 'tensor': 'head.bias', 'indices': [0], 'values': [1.0]},
            {'tool': 'edit', 'op': 'set_entries', 'tensor': 'embed.weight', 'indices': [[0, 0]], 'values': [0.0]}):
    result = probe.step({'explanation': 'guardrail probe', 'action': bad})
    assert 'refused' in result, result
    print('Refused as expected:', result['refused'])
try:
    ws.check_edit(probe.base, {**probe.base, 'block1.weight': probe.base['block1.weight'] * 0.5})
    raise AssertionError('rescale should be refused')
except ws.Refused as error:
    print('Refused as expected:', error)
assert probe.passes == 0, 'refusals must not spend budget'

rng = np.random.default_rng(0)
random_edit = {**probe.base}
d = rng.normal(size=32); d /= np.linalg.norm(d)
random_edit['block1.weight'] = probe.base['block1.weight'] - np.outer(probe.base['block1.weight'] @ d, d)
print(f"Random-direction projection on validation: {accuracy(random_edit, probe._valset):.3f} (should stay near the broken score)")

## Writer run (paid, optional)

With `RUN_MODEL = False` you can explore the harness without inference. With `True`, each writer turn makes one `AI_COMPLETE` call; a run usually takes 10 to 25 calls. The prompt goes in as a bound parameter. The model name has to be a SQL literal, so it is checked against an allowlist first. Replies are parsed as JSON **data** and never executed.

The writer sees the task, its own notes and its 10 most recent turns. Older turns are dropped, and its `notes` field carries context forward, as in dspy-wsp's handoffs.

In [ ]:
RUN_MODEL = False
MODEL = 'openai-gpt-6-sol'   # verify availability in your region at runtime
ALLOWED_MODELS = {'openai-gpt-6-sol', 'claude-opus-4-6', 'claude-sonnet-4-5'}
assert MODEL in ALLOWED_MODELS
MAX_TOKENS = 1500

raw_log = []

def writer(prompt):
    row = session.sql(f"SELECT AI_COMPLETE('{MODEL}', ?, OBJECT_CONSTRUCT('max_tokens', {int(MAX_TOKENS)})) AS COMPLETION",
                      params=[prompt]).collect()[0]
    raw = row[0]
    raw_log.append(raw if isinstance(raw, str) and len(raw) <= 20000 else str(raw)[:20000])
    return raw

if RUN_MODEL:
    ws.run_writer(harness, writer)
    print(f"Writer calls: {len(raw_log)}, turns: {harness.turns}, passes: {harness.passes}/{harness.max_passes}")
    print('Final answer:', harness.final)
else:
    print('RUN_MODEL is False; the writer was not called.')

In [ ]:
# Transcript: what the writer said, and what the harness returned.
for t in harness.transcript:
    reply = t['reply'] or {}
    action = reply.get('action') or ({'final': reply['final']} if 'final' in reply else reply)
    print(f"--- turn {t['turn']}")
    print('  why   :', str(reply.get('explanation', ''))[:300])
    print('  action:', json.dumps(action)[:300])
    print('  result:', json.dumps(t['result'])[:300])

## Validation and patch

The hidden validation set is scored **once**: a paired comparison of the original and edited weights on the same 400 examples. Then the patch is rebuilt from its hashes and checked against the live edit. The planted direction is printed last, so you can compare the writer's edit to the ground truth.

In [ ]:
report = harness.validate(fallback_if_worse=False)
print('Validation (paired):', report['validation'])

patch = harness.patch()
print('Changed tensors:', list(patch['tensors']))
print('Fraction of weights changed:', patch['fraction_of_weights_changed'])
print('Edit log:', patch['edit_log'])
rebuilt = ws.apply_patch(harness.base, patch)
assert ws.fingerprint(rebuilt) == ws.fingerprint(harness.edited)
print('Patch round-trip verified:', patch['edited_fingerprint'])

# Ground truth, revealed only now: how much of the planted direction remains in block1's inputs?
def planted_residual(weights):
    return float(np.linalg.norm(weights['block1.weight'] @ truth['v']))
print(f"Planted-direction gain: broken {planted_residual(harness.base):.3f}, edited {planted_residual(harness.edited):.3f}, clean {planted_residual(clean):.3f}")

In [ ]:
# Optional persistence: only after a real model run, and only when the table exists.
SAVE_RESULT = False
if SAVE_RESULT:
    import hashlib
    import uuid
    if not RUN_MODEL:
        raise ValueError('Only save a run with actual model responses')
    record = {
        'model': MODEL,
        'task_prompt_sha256': hashlib.sha256(ws.task_prompt(harness).encode()).hexdigest(),
        'base_fingerprint': harness.base_hash,
        'edited_fingerprint': patch['edited_fingerprint'],
        'validation': report['validation'],
        'edit_log': patch['edit_log'],
        'fraction_changed': patch['fraction_of_weights_changed'],
        'turns': harness.turns, 'passes': harness.passes,
        'final': harness.final,
        'transcript': harness.transcript,
        'raw_responses': raw_log,
        'runtime_python': platform.python_version(),
    }
    session.sql('INSERT INTO RACO_RLM_DEMO.PUBLIC.WEIGHT_EDIT_RUNS (RUN_ID, RESULT) SELECT ?, PARSE_JSON(?)',
                params=[str(uuid.uuid4()), json.dumps(record, default=float)]).collect()

## Miles: train a policy to propose repairs

The run above uses a frontier writer at inference time. [Miles](https://github.com/radixark/miles) takes the other route: it trains an **open-weight policy** with RL, so the policy gets better at proposing edits. The student networks are never trained; only the policy's own weights change.

```text
task seed -> broken student (varied seed, nuisance feature, target class, strength)
          -> one-shot evidence prompt -> policy rollout (Miles / SGLang) -> JSON edits
          -> miles_verifier.wsp_reward: guardrails + hidden validation split -> reward in [-1, 1]
          -> Miles optimizer -> new policy weights;  disjoint held-out seeds for evaluation
```

- **Reward:** the fraction of the planted accuracy loss recovered on the instance's hidden validation split, clipped to [-1, 1]. An invalid or refused edit scores 0.
- **Anti-memorization:** the nuisance feature (`caps_ratio`, `emoji_count` or `ticket_length`) and the target class change per instance. Always projecting out `block1.weight.v0` is a weak baseline, and a wrong feature guess usually scores negative (see the controls below).
- **Splits:** training seeds `1000–5999` and held-out seeds `90000–90499` never overlap. Prompt metadata carries only the seed; the reward worker rebuilds the instance from it.

This notebook **does not run Miles**. Miles needs GPU workers, SGLang and an open-weight checkpoint, none of which is deployed here. It writes the datasets, measures the controls, and optionally scores an `AI_COMPLETE` model as a zero-shot baseline for a trained policy to beat.

In [ ]:
# Reward controls on held-out seeds (free, CPU only; about 1 s per instance).
import json
N_CONTROL = 12
controls = {'analytic': [], 'always_v0': [], 'wrong_feature': [], 'invalid': []}
for seed in list(ws.MILES_HELDOUT_SEEDS)[:N_CONTROL]:
    spec = ws.task_spec(seed)
    wrong = next(f for f in ws.NUISANCE_FEATURES if f != spec['nuisance'])
    edit = lambda ref: json.dumps({'explanation': 'control', 'edits': [
        {'op': 'project_out', 'tensor': 'block1.weight', 'side': 'input', 'direction': ref}]})
    controls['analytic'].append(ws.score_proposal(seed, ws.oracle_response(seed))['reward'])
    controls['always_v0'].append(ws.score_proposal(seed, edit('block1.weight.v0'))['reward'])
    controls['wrong_feature'].append(ws.score_proposal(seed, edit(f'feature:{wrong}'))['reward'])
    controls['invalid'].append(ws.score_proposal(seed, '{"edits": [{"op": "set_entries", "tensor": "head.bias", "indices": [[0]], "values": [1]}]}')['reward'])
for name, values in controls.items():
    print(f"{name:14s} mean reward {np.mean(values):+.3f}  (n={len(values)})")
assert np.mean(controls['analytic']) > np.mean(controls['always_v0']) > np.mean(controls['wrong_feature'])
assert all(v == 0.0 for v in controls['invalid'])

In [ ]:
# Write Miles datasets. Full ranges take ~1 s per instance; keep counts small in a notebook.
N_TRAIN, N_HELDOUT = 200, 50
n_train = ws.write_miles_jsonl('wsp_miles_train.jsonl', list(ws.MILES_TRAIN_SEEDS)[:N_TRAIN], 'train')
n_heldout = ws.write_miles_jsonl('wsp_miles_heldout.jsonl', list(ws.MILES_HELDOUT_SEEDS)[:N_HELDOUT], 'heldout')
print(f'Wrote {n_train} train and {n_heldout} held-out prompts')
with open('wsp_miles_train.jsonl') as f:
    first = json.loads(f.readline())
print('Metadata:', first['metadata'])
assert 'nuisance' not in json.dumps(first['metadata'])   # the answer never reaches the policy
print(first['prompt'][0]['content'][:1200])

### Miles launch flags (template, not run here)

Add these to a separately provisioned Miles recipe, and keep its model, SGLang, Ray, training and checkpoint settings. Put `weight_space.py`, `weight_space_xor.py` and `miles_verifier.py` on the workers' Python path. Each reward rebuilds one instance on CPU in about a second, and instances are cached per worker process.

```text
--prompt-data wsp_miles_train.jsonl
--input-key prompt
--metadata-key metadata
--apply-chat-template
--custom-rm-path miles_verifier.wsp_reward
--eval-prompt-data wsp_heldout wsp_miles_heldout.jsonl
```

Report held-out reward against the controls above and the zero-shot baseline below. Rising training reward alone isn't evidence of a better policy.

In [ ]:
# Zero-shot baseline (paid, optional): an AI_COMPLETE model as the policy on held-out prompts.
RUN_BASELINE = False
N_BASELINE = 10
baseline = []
if RUN_BASELINE:
    assert MODEL in ALLOWED_MODELS
    for seed in list(ws.MILES_HELDOUT_SEEDS)[:N_BASELINE]:
        h_seed, _, _ = ws.instance(seed)
        raw = session.sql(f"SELECT AI_COMPLETE('{MODEL}', ?, OBJECT_CONSTRUCT('max_tokens', {int(MAX_TOKENS)})) AS COMPLETION",
                          params=[ws.policy_prompt(h_seed)]).collect()[0][0]
        scored = ws.score_proposal(seed, raw if isinstance(raw, str) else str(raw))
        baseline.append(scored)
        print(seed, ws.task_spec(seed)['nuisance'], 'valid' if scored['valid'] else scored.get('error'), scored['reward'])
    print(f"{MODEL} zero-shot mean held-out reward: {np.mean([b['reward'] for b in baseline]):+.3f} (n={len(baseline)})")
else:
    print('RUN_BASELINE is False; no model calls.')